<a href="https://colab.research.google.com/github/waelrash1/ai-for-business-analytics/blob/main/notebooks/Block3_Lab3_Modelling_and_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelling and evaluation with Python and an AI partner

**MI6261 AI for Business Analytics, Block 3.** Dr Wael Rashwan, School of Business, Maynooth University. MSc in Business Analytics.

The operations manager's question: **when the seller hands a parcel to the carrier, will it arrive late?** The answer is yes or no, so this is a **classification** task. Late parcels get most of the bad reviews, so an early warning lets customer care act first. In Lab 4 you will deploy the model you build here.

This lab is **vibe coding with checks**. You do not type the analysis code yourself. In each task you:

1. **Plan**: read the question and decide what one result you need.
2. **Prompt**: ask Gemini or Claude for **one small code cell**, using the five headings (Context, Task, Constraints, Format, Check).
3. **Read** the code before you run it: is anything fitted on the test set? Is a leaking column used?
4. **Run** it.
5. **Verify** one number in Excel or by a second method.
6. **Explain** the result in your own words, for a manager.

The prompt help fades: tasks 1 and 2 give you the prompt; from task 3 you write it yourself.

**Rules.** Use only what is known at the moment of the decision. Split before you fit. Beat a baseline before you trust a model. Score on the test set once. Let costs choose the threshold.

**Data.** 5,000 real delivered orders from the Olist Brazilian E-Commerce Public Dataset (Kaggle, CC BY-NC-SA 4.0), cleaned with the rules of the Block 2 data cleaning lab, with product weight, distance, seller state, promised days and handling days joined from the other Olist tables.

- **Orders**: https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv

**Extra practice.** The regression half (what shipping charge to quote at checkout) is in the Block 2 modelling notebook, tasks 2 to 6.

**AI use.** You may use any AI to write code. You must read, check and explain every cell, and keep a prompt log (end of the notebook).

---
## Task 1: Load the cleaned orders, check them, add the target

**Question.** Is this table clean enough to model, and what will the model predict?

**Plan.** One cell that produces: `n_rows`, `n_late`, `n_dup_ids`, `n_missing`.

**A weak prompt:** *"Load the data and build a model."* Which model, predicting what, on data you have checked? Rerun your cleaning checks first, then name the targets.

In [1]:
import pandas as pd
orders_url = "https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv"

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: Colab. `orders_url`: 5,000 cleaned, delivered orders with freight, weight_g, distance_km, both states, handling_days, days_late and more.
Task: Load as `orders` (order_id as text). Count repeated ids, missing values, bad state codes, negative values. Add late: 1 if days_late above 0.
Constraints: pandas only; drop or change nothing.
Format: One cell; label each check.
Check: Store `n_rows`, `n_late`, `n_dup_ids` and `n_missing`.
```

**Paste the AI's code below.** Before you run it, check that it:
- The checks only count: nothing is dropped
- Each check is one of your cleaning-lab checks
- late uses > 0: on the promised day is on time

In [2]:
import pandas as pd

orders_url = "https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv"
orders = pd.read_csv(orders_url, dtype={"order_id": str})   # order_id as text
orders["late"] = (orders["days_late"] > 0).astype(int)      # 1 if late, on the promised day is on time

n_rows = len(orders)
n_late = int(orders["late"].sum())
print("rows:", n_rows)
print("late orders:", n_late)

# Check 1: repeated order ids
n_dup_ids = int(orders["order_id"].duplicated().sum())
print("repeated order ids:", n_dup_ids)

# Check 2: missing values, all columns
n_missing = int(orders.isna().sum().sum())
print("missing values:", n_missing)

# Check 3: bad state codes (anything that is not exactly two capital letters)
states = pd.concat([orders["customer_state"], orders["seller_state"]])
n_bad_states = int((~states.str.fullmatch(r"[A-Z]{2}", na=False)).sum())
print("bad state codes:", n_bad_states)

# Check 4: negative values in columns that can never be negative (days_late is allowed to be negative: early delivery)
nonneg = ["distance_km", "n_items", "price", "freight", "weight_g", "promised_days", "handling_days", "delivery_days"]
n_negative = int((orders[nonneg] < 0).sum().sum())
print("negative values:", n_negative)


rows: 5000
late orders: 340
repeated order ids: 0
missing values: 0
bad state codes: 0
negative values: 4


**Verify in Excel.**

- Late orders in olist_model_orders.csv: `=COUNTIF(N2:N5001,">0")`

In [3]:
excel_value = None   # type what Excel shows
print(n_late == excel_value)   # True means Python and Excel agree

False


**Explain.** Three checks pass, but 4 orders were handed to the carrier before they were bought: a real recording error in Olist's data. Should they stay in the models? What would you ask the data owner?

In [ ]:
my_explanation = """

"""

---
## Task 2: Classification: features known at handover, stratified split

**Question.** When the seller hands a parcel to the carrier, can we tell if it will be late?

**Plan.** One cell that produces: `n_late_test`, `base_acc`.

**A weak prompt:** *"Split the data for classification."* Without stratify, the rare late orders (about 7 in 100) can bunch in train or test, and every score after moves.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `orders`; target late; known at handover: handling_days, promised_days, distance_km, weight_g, freight, n_items, purchase_month, customer_state, seller_state.
Task: X_c and y_c; split 75/25. Accuracy of always predicting 'on time' on test.
Constraints: random_state=42, stratify=y_c; never delivery_days or days_late.
Format: One cell; late share in each part; baseline accuracy.
Check: Store `n_late_test` and `base_acc` (3 decimals).
```

**Paste the AI's code below.** Before you run it, check that it:
- stratify=y_c
- No delivery_days, days_late or review_score in X_c
- The baseline uses the test set only to score

In [7]:
# Paste the AI's code here. Read it before you run it.
from sklearn.model_selection import train_test_split

features = ["handling_days", "promised_days", "distance_km", "weight_g", "freight",
            "n_items", "purchase_month", "customer_state", "seller_state"]
X_c = orders[features]      # known at handover only
y_c = orders["late"]

X_train, X_test, y_train, y_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)

print(f"late share: train {y_train.mean():.3f}, test {y_test.mean():.3f}")

n_late_test = int(y_test.sum())
base_acc = round(1 - y_test.mean(), 3)   # always predicting "on time" is right whenever late = 0
print("baseline accuracy (always on time):", base_acc)


late share: train 0.068, test 0.068
baseline accuracy (always on time): 0.932


**Check with a second method.** Run this check: it should print True.

- Second check: train and test have almost the same late share: that is what stratify does

In [9]:
print(abs(y_train.mean() - y_test.mean()) < 0.01)   # stratify keeps the late share equal

True


**Explain.** The baseline is right 93 times in 100 and warns nobody. What does that tell you about accuracy here?

In [ ]:
my_explanation = """

"""

---
## Task 3: A business rule as a second baseline

**Question.** How well does 'the seller was slow to ship' flag late parcels?

**Plan.** One cell that produces: `rule_precision`, `rule_recall`.

**A weak prompt:** *"Is slow handling a good predictor?"* Good by what measure? Ask for precision and recall of a simple rule, so a model has something real to beat.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `Xc_test` has handling_days; `yc_test` is late.
- **Task:** Rule: flag when handling_days is 5 or more. Show the confusion matrix; compute precision and recall.
- **Constraints:** sklearn's confusion_matrix, precision_score, recall_score; no model.
- **Format:** One cell. The matrix, then two labelled numbers.
- **Check:** Store `rule_precision` and `rule_recall` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The rule, not a model
- Rows are actual, columns predicted
- Precision and recall, not only accuracy

In [11]:
# Paste the AI's code here. Read it before you run it.
from sklearn.metrics import confusion_matrix, precision_score, recall_score

rule_pred = (X_test["handling_days"] >= 5).astype(int)   # flag late when handling_days is 5 or more

print(confusion_matrix(y_test, rule_pred))   # rows: actual on time / late; columns: predicted on time / late

rule_precision = round(precision_score(y_test, rule_pred), 3)   # of the flagged orders, share really late
rule_recall = round(recall_score(y_test, rule_pred), 3)         # of the late orders, share flagged
print("precision:", rule_precision)
print("recall:", rule_recall)


[[979 186]
 [ 56  29]]
precision: 0.135
recall: 0.341


**Check with a second method.** Run this check: it should print True.

- Second check: precision by hand: true positives / all flagged

In [12]:
print(round(((rule == 1) & (y_test == 1)).sum() / rule.sum(), 3) == rule_precision)   # precision by hand

NameError: name 'rule' is not defined

**Explain.** In words a customer care manager would use, what do this precision and this recall mean?

In [ ]:
my_explanation = """

"""

---
## Task 4: Logistic regression and the confusion matrix

**Question.** Does a model beat the rule?

**Plan.** One cell that produces: `tp_20`, `recall_20`.

**A weak prompt:** *"Build a logistic regression and show the accuracy."* Accuracy hides the misses. Ask for the confusion matrix, precision and recall, at a stated threshold.

**Write your prompt.** Use the five headings. Hints:

- **Context:** The classification train and test sets; the two states are categories.
- **Task:** Pipeline: scale, one-hot both states, LogisticRegression. Flag where the probability is 0.2 or more; confusion matrix, precision, recall. Save actual and probability as clf_test.csv.
- **Constraints:** Fit on train; max_iter=1000.
- **Format:** One cell.
- **Check:** Store `tp_20` (true positives) and `recall_20` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- Scaling and encoding are learned inside the pipeline, from train only
- predict_proba, column 1
- The threshold is written down: 0.2
- Fitted on train, scored on test

In [ ]:
# Paste the AI's code here. Read it before you run it.
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

cats = ["customer_state", "seller_state"]
nums = [c for c in X_train.columns if c not in cats]

clf = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cats),
    ])),
    ("model", LogisticRegression(max_iter=1000)),
])
clf.fit(X_train, y_train)                      # fit on train only

prob = clf.predict_proba(X_test)[:, 1]          # probability of late
pred_20 = (prob >= 0.23).astype(int)              # flag at 0.2 or more

cm = confusion_matrix(y_test, pred_20)
print(cm)
print("precision:", round(precision_score(y_test, pred_20), 3))
print("recall:", round(recall_score(y_test, pred_20), 3))

tp_20 = int(cm[1, 1])                            # actual late and flagged
recall_20 = round(recall_score(y_test, pred_20), 3)

pd.DataFrame({"actual": y_test.values, "prob": prob}).to_csv("clf_test.csv", index=False)


[[1136   29]
 [  60   25]]
precision: 0.463
recall: 0.294


**Verify in Excel.**

- Open clf_test.csv: true positives at 0.2
- True positives: `=COUNTIFS(A2:A1251,1,B2:B1251,">=0.2")`

In [ ]:
excel_value = None   # type what Excel shows
print(tp_20 == excel_value)   # True means Python and Excel agree

**Explain.** Why does 0.5 make little sense when only 7 orders in 100 are late? Compare with the rule from task 3.

In [ ]:
my_explanation = """

"""

---
## Task 5: Move the threshold

**Question.** What if we warn more customers?

**Plan.** One cell that produces: `precision_10`, `recall_10`.

**A weak prompt:** *"Find the best threshold."* Best for whom? A lower threshold catches more late parcels and warns more customers whose parcel was fine.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_late` and `yc_test` from task 4.
- **Task:** For thresholds 0.5, 0.3, 0.2 and 0.1: flagged orders, precision and recall, in one table.
- **Constraints:** No refitting; the same probabilities.
- **Format:** One cell, one small table.
- **Check:** Store precision and recall at 0.1 in `precision_10` and `recall_10` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The same p_late for every threshold
- Flagged count beside precision and recall
- No model is refitted

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: a lower threshold never catches fewer late parcels

In [ ]:
print(recall_10 >= recall_20)   # a lower threshold never catches fewer late parcels

**Explain.** Customer care can send 100 warnings from these 1,250 parcels. Which threshold fits, and what recall do you get?

In [ ]:
my_explanation = """

"""

---
## Task 6: Compare models by AUC

**Question.** Which model ranks late parcels better, at every threshold?

**Plan.** One cell that produces: `auc_log`, `auc_rf`.

**A weak prompt:** *"Which model is better?"* Better at which threshold? AUC compares how well each model ranks orders, before any threshold is chosen.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_late` from the logistic regression; `prep_c`; the classification train and test sets.
- **Task:** Fit a RandomForestClassifier in a pipeline. ROC AUC of both models on test; plot both ROC curves.
- **Constraints:** n_estimators=300, min_samples_leaf=10, random_state=42.
- **Format:** One cell: one chart, two labelled numbers.
- **Check:** Store `auc_log` and `auc_rf` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- Probabilities, not 0 or 1 predictions, go into the AUC
- Both models on the same test set
- random_state on the forest

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: AUC by hand: the chance a random late parcel scores above a random on-time one

In [ ]:
print(round((p_late[yc_test.values == 1][:, None] > p_late[yc_test.values == 0]).mean() + 0.5 * (p_late[yc_test.values == 1][:, None] == p_late[yc_test.values == 0]).mean(), 3) == auc_log)   # AUC by hand

**Explain.** AUC 0.5 is a coin toss and 1.0 is perfect. Where do these models sit, and is that good enough to act on?

In [ ]:
my_explanation = """

"""

---
## Task 7: Choose the threshold from costs

**Question.** Which threshold costs the business least?

**Plan.** One cell that produces: `best_t`, `best_cost`.

**A weak prompt:** *"What threshold should we use?"* Only costs answer that. Say what a missed late parcel costs and what an unnecessary warning costs.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_rf`, `yc_test`. A late parcel nobody warned about costs 5 (a likely bad review); a warning for a parcel that arrives on time costs 1.
- **Task:** For thresholds from 0.05 to 0.95 in steps of 0.05, total cost = 5 x misses + 1 x false alarms. Plot cost against threshold; find the cheapest.
- **Constraints:** No refitting. Round thresholds to 2 decimals.
- **Format:** One cell: one chart, one labelled result.
- **Check:** Store `best_t` and `best_cost`.

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- 5 for a miss, 1 for a false alarm, as asked
- The same p_rf; no refitting
- The chart marks the cheapest threshold

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the best plan costs less than warning nobody

In [ ]:
print(best_cost < 5 * int(yc_test.sum()))   # cheaper than warning nobody

**Explain.** Theory says the best threshold is 1 / (1 + 5), about 0.17. Is yours close? Why might it differ on 1,250 test orders?

In [ ]:
my_explanation = """

"""

---
## Your results table

| Task | Model | Measure on the test set | Value | Beats the baseline? |
|---|---|---|---|---|
| 2 | Always 'on time' (baseline) | accuracy | | |
| 3 | Slow-handover rule (baseline) | precision, recall | | |
| 4 | Logistic regression, 0.2 | precision, recall | | |
| 6 | Logistic and random forest | AUC | | |
| 7 | Random forest at the cost-based threshold | total cost | | |

## Your report (about 200 words)

**Question.** The prediction, and the moment it is made.

**Features.** Why each one is known at that moment.

**Results.** Baseline and rule against the models; the threshold you chose and why.

**Limits.** What the model cannot do, and what data would help.

**AI use.** One prompt that worked well, and one thing in the AI's code you had to correct.

---
## Your prompt log

Vibe coding is only safe when you check. Fill in three rows before you leave.

| Task | The prompt that worked (short) | What the AI got wrong, or could have | How you caught it |
|---|---|---|---|
| | | | |
| | | | |
| | | | |